# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Srilaya30/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

I build the final feature vector from search, engagement, content-age, and page-performance signals. The target-related fields are kept out of the feature vector. Missing numerical values are handled using median imputation.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import pandas as pd
import numpy as np

repo_root = Path("/content/flyrank-ml-internship")

if not repo_root.exists():
    !git clone -q https://github.com/Srilaya30/flyrank-ml-internship.git

data_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"

print("Dataset exists:", data_path.exists())

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)

feature_cols = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

X = df[feature_cols].copy()

print("Number of features:", len(feature_cols))
print("Feature matrix shape:", X.shape)

missing_before = X.isna().sum().sum()

X = X.fillna(X.median(numeric_only=True))

missing_after = X.isna().sum().sum()

print("Missing values before fill:", missing_before)
print("Missing values after fill:", missing_after)

Dataset exists: True
Dataset shape: (30000, 44)
Number of features: 29
Feature matrix shape: (30000, 29)
Missing values before fill: 22927
Missing values after fill: 0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

The features represent search demand, traffic, engagement, content age, and page-performance signals. Numerical missing values are filled using the median of the available data. The features are intended to represent information available before the prediction decision; target-derived trend fields are not included.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_notes = pd.DataFrame({
    "feature": feature_cols,
    "dtype": X.dtypes.astype(str).values,
    "missing_after_fill": X.isna().sum().values
})

print("Feature notes:")
display(feature_notes)

print("\nAll missing values handled:",
      X.isna().sum().sum() == 0)

Feature notes:


,feature,dtype,missing_after_fill
0,search_volume,float64,0
1,competition,float64,0
2,cpc,float64,0
3,word_count,float64,0
4,char_count,float64,0
5,impressions_90d,int64,0
6,clicks_90d,int64,0
7,pageviews_90d,int64,0
8,sessions_90d,int64,0
9,users_90d,int64,0



All missing values handled: True


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I checked the final feature vector for fields that directly define the target or reveal the outcome. I specifically exclude trend fields used to construct the declining label. I also check the feature names for obvious future/outcome-related terms.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Target definition
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

target_related = {
    "trend_direction",
    "trend_pct",
    "is_declining_label"
}

leakage_columns = [
    col for col in feature_cols
    if col in target_related
]

future_keywords = [
    "future",
    "next_",
    "outcome",
    "target",
    "label"
]

future_like_columns = [
    col for col in feature_cols
    if any(word in col.lower() for word in future_keywords)
]

print("Target-related columns found in features:")
print(leakage_columns)

print("\nFuture/outcome-like feature names:")
print(future_like_columns)

print("\nLeakage check:",
      "PASS" if len(leakage_columns) == 0
      else "REVIEW REQUIRED")

Target-related columns found in features:
[]

Future/outcome-like feature names:
[]

Leakage check: PASS


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

I excluded fields that directly define the target or could reveal the outcome. I also avoid using client-identifying information as a predictive feature. These exclusions reduce the risk of target leakage and keep the public analysis focused on decision-support signals.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded_fields = pd.DataFrame({
    "field": [
        "trend_direction",
        "trend_pct",
        "is_declining_label",
        "client_id"
    ],
    "reason": [
        "Used to define the declining target.",
        "Direct trend/outcome information.",
        "The prediction target itself.",
        "Used for grouping/validation, not prediction."
    ]
})

display(excluded_fields)

,field,reason
0,trend_direction,Used to define the declining target.
1,trend_pct,Direct trend/outcome information.
2,is_declining_label,The prediction target itself.
3,client_id,"Used for grouping/validation, not prediction."


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [9]:
print("W03 SELF-CHECK")
print("=" * 40)

print("Dataset rows:", len(df))
print("Original columns:", len(df.columns))
print("Final features:", len(feature_cols))
print("Feature matrix:", X.shape)
print("Missing values after handling:",
      X.isna().sum().sum())
print("Leakage columns:",
      leakage_columns)
print("Leakage status:",
      "PASS" if len(leakage_columns) == 0
      else "REVIEW REQUIRED")

assert len(feature_cols) == 29
assert X.shape[1] == 29
assert X.isna().sum().sum() == 0
assert len(leakage_columns) == 0

print("\nW03 CHECK: PASS")

W03 SELF-CHECK
Dataset rows: 30000
Original columns: 45
Final features: 29
Feature matrix: (30000, 29)
Missing values after handling: 0
Leakage columns: []
Leakage status: PASS

W03 CHECK: PASS
